# Country GDP Growth Rate

## Problem Statement

You are a data engineer on the economics research team at Amazon.

GDP figures arrive from two separate vendor feeds, `gdp_df1` and `gdp_df2`, each containing part of the country-year GDP history. Leadership requires a unified year-over-year GDP growth series for each country.

## Input Tables

### gdp_df1

| Column Name | Data Type |
|------------|-----------|
| country | VARCHAR |
| year | INT |
| gdp | DECIMAL |

### gdp_df2

| Column Name | Data Type |
|------------|-----------|
| country | VARCHAR |
| year | INT |
| gdp | DECIMAL |

## Requirements

- Combine records from both GDP feeds.
- Compare each country's GDP with its previous available year's GDP.
- Calculate year-over-year GDP growth rates.
- The earliest year for a country should have a NULL growth rate.
- Return country as `Country`.
- Return year as `Year`.
- Sort results by country ascending and year ascending.
- Return results matching the required output schema and order.

## Output Columns

| Column Name |
|------------|
| Country |
| Year |
| GDP_growth_rate |

## Sample Input

### gdp_df1

| country | year | gdp |
|---------|------|---------|
| USA | 2018 | 20544.34 |
| USA | 2019 | 21427.70 |
| China | 2018 | 13894.04 |

### gdp_df2

| country | year | gdp |
|---------|------|---------|
| China | 2019 | 14402.72 |
| India | 2018 | 2713.61 |
| India | 2019 | 2868.93 |

## Sample Output

| Country | Year | GDP_growth_rate |
|----------|------|----------------|
| China | 2018 | NULL |
| China | 2019 | 3.66 |
| India | 2018 | NULL |
| India | 2019 | 5.72 |
| USA | 2018 | NULL |
| USA | 2019 | 4.3 |

## Expected Output Schema

| Column Name | Data Type |
|------------|-----------|
| Country | STRING |
| Year | INT |
| GDP_growth_rate | DECIMAL |

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import *

# gdp_df1
gdp_df1_schema = StructType([
    StructField("country", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("gdp", DoubleType(), True)
])

gdp_df1_data = [
    ("USA", 2018, 20544.34),
    ("USA", 2019, 21427.70),
    ("China", 2018, 13894.04)
]

gdp_df1_df = spark.createDataFrame(
    gdp_df1_data,
    schema=gdp_df1_schema
)

# gdp_df2
gdp_df2_schema = StructType([
    StructField("country", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("gdp", DoubleType(), True)
])

gdp_df2_data = [
    ("China", 2019, 14402.72),
    ("India", 2018, 2713.61),
    ("India", 2019, 2868.93)
]

gdp_df2_df = spark.createDataFrame(
    gdp_df2_data,
    schema=gdp_df2_schema
)

In [0]:
gdp_df = gdp_df1_df.unionByName(gdp_df2_df)

result_df = (
    gdp_df.withColumn(
        "previous_gdp", lag("gdp").over(Window.partitionBy("country").orderBy("year"))
    )
    .withColumn(
        "gpd_growth_rate",
        (col("gdp") - col("previous_gdp")) * 100 / col("previous_gdp"),
    )
    .select(
        col("country"),
        col("year"),
        coalesce(round(col("gpd_growth_rate"), 2).cast("string"), lit("")).alias(
            "gpd_growth_rate"
        )
    )
)
display(result_df)